# Двор, карточки Pokemon и накопление преимущества

Модель ниже иллюстрирует не тезис `богатые лучше торгуются`, а более узкий механизм: больше карточек означает больше независимых возможностей обмена и доступ к более крупным сделкам. Одна конкретная возможность обмена устроена одинаково для всех.

Ключевой ход модели: каждая карточка независимо может породить торговый lead. Если у ребенка 3 карточки, у него 3 источника leads; если 40 карточек, их 40. Это не выбор ребенка с вероятностью, пропорциональной богатству, а явная генерация возможностей от портфеля карточек.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from pokemon_yard_model import (
    make_initial_distribution,
    simulate_exchanges,
    describe_distribution,
    group_shares,
    lorenz_curve,
)

plt.style.use("seaborn-v0_8-whitegrid")
plt.rcParams["figure.figsize"] = (12, 5)
plt.rcParams["axes.titlesize"] = 14
plt.rcParams["axes.labelsize"] = 11

## 1. Изначальное распределение

Берем 100 детей. Число карточек на старте генерируется нормальным распределением с отсечением слева в ноль: отрицательных карточек быть не может, поэтому часть детей может получить 0.

In [ ]:
initial = make_initial_distribution(
    n_children=100,
    mean_cards=20,
    std_cards=12,
    seed=7,
)

summary_initial = pd.Series(describe_distribution(initial), name="initial")
summary_initial

In [ ]:
order = np.argsort(initial)
colors = np.array(["#d14a4a"] * 20 + ["#4c78a8"] * 60 + ["#3b9b67"] * 20)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].bar(np.arange(len(initial)), initial[order], color=colors, width=0.9)
axes[0].set_title("Старт: дети отсортированы по числу карточек")
axes[0].set_xlabel("Ранг на старте")
axes[0].set_ylabel("Карточки")

axes[1].hist(initial, bins=18, color="#4c78a8", edgecolor="white")
axes[1].axvline(initial.mean(), color="black", linestyle="--", label=f"mean = {initial.mean():.1f}")
axes[1].set_title("Форма стартового распределения")
axes[1].set_xlabel("Карточки у ребенка")
axes[1].set_ylabel("Число детей")
axes[1].legend()

plt.tight_layout()

## 2. Модель обменов

На каждом шаге:

1. Каждая уже имеющаяся карточка независимо может создать возможность обмена с вероятностью `card_lead_probability`.
2. Одна возможность обмена имеет одинаковое распределение `spread` для всех детей. У богатого ребенка нет лучшей вероятности выиграть одну конкретную сделку.
3. Если spread положительный, ребенок может забрать часть карточек у контрагента.
4. Размер сделки ограничен бюджетом инициатора: у кого мало карточек, тот физически не может участвовать в крупных обменах.
5. Общее число карточек сохраняется.

In [ ]:
result = simulate_exchanges(
    initial,
    steps=180,
    card_lead_probability=0.006,
    stake_fraction=0.08,
    max_transfer=4,
    spread_to_cards=0.35,
    seed=11,
)

final = result.history[-1]
pd.DataFrame({
    "initial": summary_initial,
    "final": pd.Series(describe_distribution(final)),
})

In [ ]:
total_leads = result.opportunity_history.sum(axis=0)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].scatter(initial, total_leads, s=50, alpha=0.78, color="#4c78a8")
axes[0].set_title("Возможности возникают от карточек, а не от 'таланта'")
axes[0].set_xlabel("Карточки на старте")
axes[0].set_ylabel("Всего торговых leads за симуляцию")

axes[1].scatter(initial, final - initial, s=50, alpha=0.78, color="#d14a4a")
axes[1].axhline(0, color="black", linewidth=1)
axes[1].set_title("Итоговое изменение против стартовой позиции")
axes[1].set_xlabel("Карточки на старте")
axes[1].set_ylabel("Финал минус старт")

plt.tight_layout()

## 3. Перетекание карточек по стартовым группам

Делим детей не по финалу, а по стартовой позиции: нижние 20%, средние 60%, верхние 20%. Это важно: так видно, что происходит с теми, кому изначально повезло или не повезло.

In [ ]:
shares = group_shares(result.history, initial)
steps = np.arange(result.history.shape[0])

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(steps, shares["bottom_20"] * 100, color="#d14a4a", label="нижние 20% старта")
axes[0].plot(steps, shares["middle_60"] * 100, color="#4c78a8", label="средние 60%")
axes[0].plot(steps, shares["top_20"] * 100, color="#3b9b67", label="верхние 20% старта")
axes[0].set_title("Доля всех карточек у стартовых групп")
axes[0].set_xlabel("Шаг")
axes[0].set_ylabel("Доля всех карточек, %")
axes[0].legend()

gini_series = [describe_distribution(row)["gini"] for row in result.history]
axes[1].plot(steps, gini_series, color="black")
axes[1].set_title("Неравенство растет в ходе обменов")
axes[1].set_xlabel("Шаг")
axes[1].set_ylabel("Gini")

plt.tight_layout()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].bar(np.arange(len(initial)), initial[order], color=colors, alpha=0.45, label="старт")
axes[0].bar(np.arange(len(final)), final[order], color=colors, alpha=0.9, width=0.55, label="финал")
axes[0].set_title("Те же дети, отсортированные по стартовой позиции")
axes[0].set_xlabel("Ранг на старте")
axes[0].set_ylabel("Карточки")
axes[0].legend()

x0, y0 = lorenz_curve(initial)
x1, y1 = lorenz_curve(final)
axes[1].plot([0, 1], [0, 1], color="gray", linestyle="--", label="равенство")
axes[1].plot(x0, y0, color="#4c78a8", label="старт")
axes[1].plot(x1, y1, color="#d14a4a", label="финал")
axes[1].set_title("Кривая Лоренца")
axes[1].set_xlabel("Доля детей, от бедных к богатым")
axes[1].set_ylabel("Доля карточек")
axes[1].legend()

plt.tight_layout()

## 4. Manim-визуализация процесса

Файл сцены уже лежит рядом: `manim_card_flow.py`. Он использует ту же модель и показывает детей в порядке стартового ранга. Желтые дуги показывают крупные потоки карточек от более левой стартовой позиции к более правой внутри окна анимации.

В Binder важно запускать Manim без флага `-p`: этот флаг пытается открыть видео через системный viewer (`xdg-open`), которого в Binder обычно нет.

Команда для Binder или другого headless-окружения:

```bash
manim -ql manim_card_flow.py CardFlowScene
```

Локально, если хотите сразу открыть preview после рендера, можно использовать:

```bash
manim -pql manim_card_flow.py CardFlowScene
```

In [ ]:
from IPython.display import Video, display

# Binder/headless-safe: no -p flag, because -p calls xdg-open after rendering.
!manim -ql manim_card_flow.py CardFlowScene

video_path = "media/videos/manim_card_flow/480p15/CardFlowScene.mp4"
display(Video(video_path, embed=True, html_attributes="controls"))